# Full-Scale Training: Hybrid Edge Transformer
All 600K COCO captions, 4-layer model, train until convergence.
Saves checkpoints, tracks loss, generates samples.

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, math, time, os, json, zipfile
import matplotlib.pyplot as plt

## 1. Data: All COCO Captions

In [ ]:
zip_path = os.path.expanduser('~/Desktop/918822019.github.io/data/coco/PAI/COCO2017/annotations_trainval2017.zip')
with zipfile.ZipFile(zip_path, 'r') as z:
    with z.open('annotations/captions_train2017.json') as f: train = json.load(f)
    with z.open('annotations/captions_val2017.json') as f: val = json.load(f)
caps = [a['caption'].lower() for a in train['annotations'] + val['annotations']]
print(f"Total captions: {len(caps):,}")

chars = sorted(list(set(''.join(caps))))
v = len(chars) + 2
BOS, EOS = v-2, v-1
stoi = {c:i+2 for i,c in enumerate(chars)}
itos = {i+2:c for i,c in enumerate(chars)}
itos[BOS], itos[EOS] = '[BOS]', '[EOS]'
stoi['[BOS]'], stoi['[EOS]'] = BOS, EOS
def enc(s): return [BOS] + [stoi[c] for c in s if c in stoi] + [EOS]
print(f"Vocab: {v}")

all_tokens = []
for cap in caps:
    all_tokens.extend(enc(cap))
data = torch.tensor(all_tokens, dtype=torch.long)
n1 = int(0.9 * len(data))
train_data, val_data = data[:n1], data[n1:]
print(f"Tokens: train {len(train_data):,}, val {len(val_data):,}")
del caps, all_tokens

def get_batch(data, sl=128, bs=32):
    ix = torch.randint(len(data)-sl-1, (bs,))
    x = torch.stack([data[i:i+sl] for i in ix])
    y = torch.stack([data[i+1:i+sl+1] for i in ix])
    return x, y

## 2. Multi-Layer Model

In [ ]:
def linear_attn_seq(q, k, v, decay=0.99, eps=1e-6):
    B, H, T, D = q.shape
    qf, kf, vf = F.elu(q)+1, F.elu(k)+1, F.elu(v)+1
    S = torch.zeros(B, H, D, D, device=q.device)
    z = torch.zeros(B, H, D, device=q.device)
    outs = []
    for i in range(T):
        S = decay * S + kf[:,:,i:i+1].transpose(-2,-1) @ vf[:,:,i:i+1]
        z = decay * z + kf[:,:,i]
        out = (qf[:,:,i:i+1] @ S) / (qf[:,:,i:i+1] @ z.unsqueeze(-1)).clamp(min=eps)
        outs.append(out)
    return torch.cat(outs, -2)

class RoPE(nn.Module):
    def __init__(self, dim, base=10000.0):
        super().__init__()
        inv_freq = 1.0/(base**(torch.arange(0,dim,2).float()/dim))
        self.register_buffer('inv_freq', inv_freq)
    def forward(self, x, offset=0):
        t = torch.arange(offset, offset+x.shape[-2], device=x.device).type_as(self.inv_freq)
        freqs = t.unsqueeze(-1) @ self.inv_freq.unsqueeze(0)
        return torch.cat([freqs, freqs], -1)
def apply_rope(x, emb):
    s = x.shape[-1]//2
    xc = torch.view_as_complex(x.reshape(*x.shape[:-1],s,2).contiguous())
    ec = torch.view_as_complex(emb.reshape(*emb.shape[:-1],s,2).contiguous())
    return torch.view_as_real(xc*ec).reshape(*x.shape[:-1],-1)

In [ ]:
class HybridLayer(nn.Module):
    def __init__(self, dim=256, hd=64, nh=4):
        super().__init__()
        self.dim, self.hd, self.nh = dim, hd, nh
        self.csa_k = nn.Linear(dim, hd, 0); self.csa_v = nn.Linear(dim, hd, 0)
        self.csa_q = nn.Linear(dim, hd, 0); self.csa_o = nn.Linear(hd, dim, 0)
        self.hca_k = nn.Linear(dim, hd, 0); self.hca_v = nn.Linear(dim, hd, 0)
        self.hca_q = nn.Linear(dim, hd, 0); self.hca_o = nn.Linear(hd, dim, 0)
        self.swa_q = nn.Linear(dim, dim, 0); self.swa_k = nn.Linear(dim, dim, 0)
        self.swa_v = nn.Linear(dim, dim, 0); self.swa_o = nn.Linear(dim, dim, 0)
        self.norm1 = nn.LayerNorm(dim); self.norm2 = nn.LayerNorm(dim)
        self.ffn = nn.Sequential(nn.Linear(dim, dim*4), nn.GELU(), nn.Linear(dim*4, dim))
        self.gate = nn.Parameter(torch.ones(3))
    def forward(self, h, rope_emb):
        B, T, _ = h.shape
        qc = self.csa_q(h).view(B,T,1,self.hd).transpose(1,2)
        kc = self.csa_k(h).view(B,T,1,self.hd).transpose(1,2)
        vc = self.csa_v(h).view(B,T,1,self.hd).transpose(1,2)
        oc = self.csa_o(linear_attn_seq(qc,kc,vc,0.99).squeeze(1))
        qh = self.hca_q(h).view(B,T,1,self.hd).transpose(1,2)
        kh = self.hca_k(h).view(B,T,1,self.hd).transpose(1,2)
        vh = self.hca_v(h).view(B,T,1,self.hd).transpose(1,2)
        oh = self.hca_o(linear_attn_seq(qh,kh,vh,0.999).squeeze(1))
        qs = self.swa_q(h).view(B,T,self.nh,self.hd).transpose(1,2)
        ks = self.swa_k(h).view(B,T,self.nh,self.hd).transpose(1,2)
        vs = self.swa_v(h).view(B,T,self.nh,self.hd).transpose(1,2)
        emb = rope_emb[:T] if rope_emb.shape[-2] >= T else rope_emb
        ks = apply_rope(ks, emb); qs = apply_rope(qs, emb)
        os_ = F.scaled_dot_product_attention(qs, ks, vs, is_causal=True)
        os_ = self.swa_o(os_.transpose(1,2).reshape(B,T,-1))
        g = F.softmax(self.gate, 0)
        h = self.norm1(h + g[0]*oc + g[1]*oh + g[2]*os_)
        return h + self.ffn(self.norm2(h))

class EdgeTransformer(nn.Module):
    def __init__(self, n_layers=4, dim=256, hd=64, nh=4, v=65):
        super().__init__()
        self.embed = nn.Embedding(v, dim)
        self.rope = RoPE(hd)
        self.layers = nn.ModuleList([HybridLayer(dim, hd, nh) for _ in range(n_layers)])
        self.lm_head = nn.Linear(dim, v, 0)
        for m in self.modules():
            if isinstance(m, nn.Linear): nn.init.normal_(m.weight, 0.0, 0.02)
            if isinstance(m, nn.Linear) and m.bias is not None: nn.init.zeros_(m.bias)
            if isinstance(m, nn.Embedding): nn.init.normal_(m.weight, 0.0, 0.02)
    def forward(self, x):
        h = self.embed(x)
        rope_emb = self.rope(h)
        for layer in self.layers:
            h = layer(h, rope_emb)
        return self.lm_head(h)

## 3. Training (with checkpoints)

In [ ]:
m = EdgeTransformer(n_layers=4, dim=256, hd=64, nh=4, v=v)
print(f"Params: {sum(p.numel() for p in m.parameters()):,}")
opt = torch.optim.AdamW(m.parameters(), lr=3e-4)
steps = 5000; log_every = 500; save_every = 1000
tl, vl = [], []; t0 = time.time(); best_loss = float('inf')
ckpt_dir = 'checkpoints'; os.makedirs(ckpt_dir, exist_ok=True)

for step in range(steps):
    m.train(); x, y = get_batch(train_data, 128, 32)
    loss = F.cross_entropy(m(x).reshape(-1, v), y.reshape(-1))
    opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(m.parameters(), 1.0); opt.step()
    if step % log_every == 0:
        m.eval()
        with torch.no_grad():
            xv, yv = get_batch(val_data, 128, 32)
            lv = F.cross_entropy(m(xv).reshape(-1, v), yv.reshape(-1))
        tl.append(loss.item()); vl.append(lv.item())
        ppl = math.exp(lv.item())
        print(f"step {step:>5} | train {loss.item():.4f} | val {lv.item():.4f} | ppl {ppl:.2f} | {time.time()-t0:.0f}s")
        if lv.item() < best_loss:
            best_loss = lv.item()
            torch.save(m.state_dict(), f'{ckpt_dir}/best.pt')
    if step > 0 and step % save_every == 0:
        torch.save({'model': m.state_dict(), 'opt': opt.state_dict(), 'step': step},
                   f'{ckpt_dir}/step_{step}.pt')

torch.save(m.state_dict(), f'{ckpt_dir}/final.pt')
print(f"Training done. Best val loss: {best_loss:.4f}, ppl: {math.exp(best_loss):.2f}")

## 4. Results

In [ ]:
plt.figure(figsize=(10,3))
plt.subplot(131); plt.plot(tl, label='train'); plt.plot(vl, label='val')
plt.xlabel('step'); plt.ylabel('loss'); plt.legend()
plt.subplot(132); plt.plot([math.exp(l) for l in vl])
plt.xlabel('step'); plt.ylabel('perplexity')
plt.subplot(133)
plt.plot([v.item() if isinstance(v, torch.Tensor) else v for v in vl])
plt.xlabel('step'); plt.ylabel('val loss')
plt.tight_layout(); plt.show()
print(f"Best: loss {best_loss:.4f}, ppl {math.exp(best_loss):.2f}")

In [ ]:
def generate(m, prompt='a', max_n=100, temp=0.8):
    m.eval(); ix = torch.tensor([[BOS] + [stoi.get(c, BOS) for c in prompt]], dtype=torch.long)
    with torch.no_grad():
        for _ in range(max_n):
            logits = m(ix[:, -512:])
            p = F.softmax(logits[:, -1] / temp, -1)
            ix = torch.cat([ix, torch.multinomial(p, 1)], -1)
            if ix[0, -1].item() == EOS:
                break
    return ''.join([itos.get(i.item(), '?') for i in ix[0]])

m.load_state_dict(torch.load(f'{ckpt_dir}/best.pt'))
for prompt in ['a', 'the', 'an', 'two']:
    print(f'{prompt}: {generate(m, prompt)}')
    print()